# Tune BERT with Manual Sarcasm Rationales — Kaggle Edition

This notebook compares a toxicity-only BERT baseline with a multitask BERT that learns sarcasm and manually annotated rationale spans. It adds a **validation-only auxiliary-loss sweep** to seek a better balance between classification macro-F1 and rationale quality.

**Important:** this is an extension inspired by the research paper, not an exact reproduction. The held-out test set is not used to choose loss weights or checkpoints. Attach the manual annotation CSV using Kaggle's **Add Input** panel.


## 1. Install only the required NLP package versions

Kaggle already includes a managed NumPy/SciPy/Pandas/PyTorch stack. **Do not force-reinstall those packages**, because that can break Kaggle's preinstalled CUDA and data-science dependencies. This cell installs Transformers and Tokenizers only.


In [7]:
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3"


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [8]:
import os, re, random, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
SEEDS = [42, 43, 44]   # For a quick smoke test, change to [42]
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
# Defaults; the validation sweep explicitly passes each candidate.
SARCASM_LOSS_WEIGHT = 0.1
RATIONALE_LOSS_WEIGHT = 0.01
# Compact grid to keep Kaggle runtime manageable.
TUNING_CONFIGS = [
    {"name": "sar_005_rat_0005", "sarcasm_weight": 0.05, "rationale_weight": 0.005},
    {"name": "sar_005_rat_001",  "sarcasm_weight": 0.05, "rationale_weight": 0.01},
    {"name": "sar_01_rat_0005",  "sarcasm_weight": 0.10, "rationale_weight": 0.005},
    {"name": "sar_01_rat_001",   "sarcasm_weight": 0.10, "rationale_weight": 0.01},
    {"name": "sar_01_rat_002",   "sarcasm_weight": 0.10, "rationale_weight": 0.02},
]
NUM_SUPERVISED_HEADS = 6
NUM_WORKERS = 0

os.makedirs("/kaggle/working", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Find the annotated CSV attached through Kaggle's Add Input panel.
import glob
CSV_CANDIDATES = glob.glob("/kaggle/input/**/*.csv", recursive=True)
CSV_CANDIDATES = [p for p in CSV_CANDIDATES if "sarcasm_manual" in os.path.basename(p).lower()]

if CSV_CANDIDATES:
    CSV_PATH = CSV_CANDIDATES[0]
    print("Using CSV:", CSV_PATH)
else:
    raise FileNotFoundError(
        "Could not find a CSV named like 'sarcasm_manual...csv' under /kaggle/input. "
        "Use Kaggle's Add Input to attach the dataset, or change CSV_PATH manually to its /kaggle/input/... path."
    )

df_raw = pd.read_csv(CSV_PATH)
print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())


Device: cuda
Tesla T4
Using CSV: /kaggle/input/datasets/parthsingh1384/sarcasm-manual-3199-3-csv/sarcasm_manual_3199 (2).csv
Raw shape: (3199, 5)
Columns: ['post_id', 'text', 'label', 'sarcasm_label', 'sarcasm_rationale']


,post_id,text,label,sarcasm_label,sarcasm_rationale
0,13517786_gab,they used to have cracker and honky now they h...,offensive,sarcasm,does this mean i can say nigger faggot kike sa...
1,1179019319717568512_twitter,now playing on dnaradio lil duval ft ball gree...,normal,non-sarcasm,NaN
2,1179075099569393666_twitter,<user> there is need for awsreness campaign to...,normal,non-sarcasm,NaN
3,14049816_gab,gatewaypundit is pure fake news it just clickb...,offensive,non-sarcasm,NaN
4,14537881_gab,a majority of us votes against muslim immigrat...,hatespeech,non-sarcasm,NaN


## 2. Validate and inspect the uploaded annotations

The paper uses three toxicity classes. The ten `undecided` entries are excluded from the 3-class classifier, matching the task definition rather than silently mapping them to another class. The sarcasm labels remain a separate binary target.

In [9]:
REQUIRED = ["post_id", "text", "label", "sarcasm_label", "sarcasm_rationale"]
missing = [c for c in REQUIRED if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df_raw.copy()
df = df.dropna(subset=["post_id", "text", "label", "sarcasm_label"]).copy()
df["post_id"] = df["post_id"].astype(str)
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(str).str.lower().str.strip()
df["sarcasm_label"] = df["sarcasm_label"].astype(str).str.lower().str.strip()

# De-duplicate by post ID before splitting to prevent leakage.
df = df.drop_duplicates(subset=["post_id"]).copy()

label2id = {"hatespeech": 0, "offensive": 1, "normal": 2}
sarcasm2id = {"non-sarcasm": 0, "sarcasm": 1}

print("Toxicity labels before filtering:\n", df["label"].value_counts(dropna=False))
print("\nSarcasm labels:\n", df["sarcasm_label"].value_counts(dropna=False))

df = df[df["label"].isin(label2id) & df["sarcasm_label"].isin(sarcasm2id)].copy()
df["label_id"] = df["label"].map(label2id).astype(int)
df["sarcasm_id"] = df["sarcasm_label"].map(sarcasm2id).astype(int)
df = df.reset_index(drop=True)

print("\nUsable rows:", len(df))
display(pd.crosstab(df["label"], df["sarcasm_label"], margins=True))
print("Missing rationale:", df["sarcasm_rationale"].isna().sum())


Toxicity labels before filtering:
 label
normal        1552
hatespeech    1193
offensive      444
undecided       10
Name: count, dtype: int64

Sarcasm labels:
 sarcasm_label
non-sarcasm    2751
sarcasm         448
Name: count, dtype: int64

Usable rows: 3189


sarcasm_label,non-sarcasm,sarcasm,All
label,,,
hatespeech,1046,147,1193
normal,1322,230,1552
offensive,375,69,444
All,2743,446,3189


Missing rationale: 2743


## 3. Stratified train/validation/test split

Use a joint stratum of toxicity label and sarcasm label so the smaller sarcastic-offensive and sarcastic-hate groups are represented in each split. The test set is held out and must not be used to tune hyperparameters.

In [10]:
df["joint_stratum"] = df["label"].astype(str) + "__" + df["sarcasm_label"].astype(str)

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["joint_stratum"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["joint_stratum"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} rows")
    print(pd.crosstab(part["label"], part["sarcasm_label"]))
    print()


Train: 2551 rows
sarcasm_label  non-sarcasm  sarcasm
label                              
hatespeech             837      118
normal                1057      184
offensive              300       55

Validation: 319 rows
sarcasm_label  non-sarcasm  sarcasm
label                              
hatespeech             104       15
normal                 132       23
offensive               38        7

Test: 319 rows
sarcasm_label  non-sarcasm  sarcasm
label                              
hatespeech             105       14
normal                 133       23
offensive               37        7



## 4. Tokenization and rationale alignment

Rationales are aligned by exact case-insensitive substring matching. A rationale that cannot be matched is marked invalid for rationale loss and rationale metrics, but its text remains usable for classification and sarcasm prediction. Inspect the alignment rate; if it is low, normalize annotations or manually review mismatches before drawing conclusions.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def find_rationale_span(text, rationale):
    if rationale is None or pd.isna(rationale):
        return None
    rationale = str(rationale).strip()
    if not rationale:
        return None
    # First try literal case-insensitive matching.
    start = text.lower().find(rationale.lower())
    if start >= 0:
        return (start, start + len(rationale))
    # Whitespace-normalized fallback, preserving an approximate span in original text.
    norm_text = re.sub(r"\\s+", " ", text.lower())
    norm_rat = re.sub(r"\\s+", " ", rationale.lower())
    start_norm = norm_text.find(norm_rat)
    if start_norm >= 0:
        # Map normalized positions back approximately to original character positions.
        char_map = []
        prev_space = False
        for i, ch in enumerate(text.lower()):
            if ch.isspace():
                if not prev_space:
                    char_map.append(i)
                prev_space = True
            else:
                char_map.append(i)
                prev_space = False
        if start_norm < len(char_map) and start_norm + len(norm_rat) - 1 < len(char_map):
            return (char_map[start_norm], char_map[start_norm + len(norm_rat) - 1] + 1)
    return None

class HateSarcasmDataset(Dataset):
    def __init__(self, frame):
        self.rows = []
        for _, row in frame.iterrows():
            text = row["text"]
            enc = tokenizer(
                text, max_length=MAX_LEN, truncation=True, padding="max_length",
                return_offsets_mapping=True, return_tensors="pt"
            )
            offsets = enc.pop("offset_mapping")[0].numpy()
            rationale_mask = np.zeros(MAX_LEN, dtype=np.float32)
            span = find_rationale_span(text, row.get("sarcasm_rationale"))

            valid = False
            if int(row["sarcasm_id"]) == 1 and span is not None:
                start, end = span
                for j, (a, b) in enumerate(offsets):
                    # Ignore special tokens and padding (offsets usually (0,0)).
                    if b > a and a < end and b > start:
                        rationale_mask[j] = 1.0
                valid = rationale_mask.sum() > 0

            self.rows.append({
                "input_ids": enc["input_ids"][0],
                "attention_mask": enc["attention_mask"][0],
                "label": int(row["label_id"]),
                "sarcasm": int(row["sarcasm_id"]),
                "rationale_mask": torch.tensor(rationale_mask, dtype=torch.float32),
                "rationale_valid": bool(valid),
                "text": text,
            })

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

train_ds = HateSarcasmDataset(train_df)
val_ds = HateSarcasmDataset(val_df)
test_ds = HateSarcasmDataset(test_df)

def alignment_summary(dataset, name):
    sarcastic = sum(x["sarcasm"] == 1 for x in dataset)
    valid = sum(x["rationale_valid"] for x in dataset)
    print(f"{name}: sarcastic rows={sarcastic}, matched sarcasm rationales={valid}, "
          f"match rate={valid / max(sarcastic, 1):.3f}")

alignment_summary(train_ds, "Train")
alignment_summary(val_ds, "Validation")
alignment_summary(test_ds, "Test")

def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS)



Train: sarcastic rows=357, matched sarcasm rationales=354, match rate=0.992
Validation: sarcastic rows=45, matched sarcasm rationales=43, match rate=0.956
Test: sarcastic rows=44, matched sarcasm rationales=43, match rate=0.977


### DataLoader compatibility fix
The dataset deliberately does **not** return `rationale_span` in each sample. Unmatched rationales are represented by `rationale_valid=False` and an all-zero `rationale_mask`; returning `None` in a dictionary field causes PyTorch's default DataLoader collator to fail.


## 5. Model and losses

Both systems share the same BERT backbone and 3-class head. The proposed model additionally learns sarcasm classification and is supervised to place `[CLS]` attention on manually annotated sarcasm rationale tokens. The rationale loss is masked to examples with successfully aligned sarcasm rationales.

This is a practical extension, not the paper's exact combined rationale vector: the original HateXplain rationales are not present in your CSV.

In [12]:
class MultiTaskBERT(nn.Module):
    def __init__(self, model_name=MODEL_NAME, supervised_heads=NUM_SUPERVISED_HEADS):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name, attn_implementation="eager"
        )
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.toxicity_head = nn.Linear(hidden, 3)
        self.sarcasm_head = nn.Linear(hidden, 2)
        self.supervised_heads = min(supervised_heads, self.encoder.config.num_attention_heads)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True
        )
        pooled = self.dropout(out.last_hidden_state[:, 0])
        toxicity_logits = self.toxicity_head(pooled)
        sarcasm_logits = self.sarcasm_head(pooled)
        # [batch, heads, sequence], CLS attention from final layer.
        cls_attention = out.attentions[-1][:, :self.supervised_heads, 0, :]
        # Zero out padding and renormalize each head.
        cls_attention = cls_attention * attention_mask.unsqueeze(1)
        cls_attention = cls_attention / cls_attention.sum(-1, keepdim=True).clamp_min(1e-8)
        return toxicity_logits, sarcasm_logits, cls_attention

def rationale_loss_fn(attention, rationale_targets, valid_flags):
    losses = []
    for i, valid in enumerate(valid_flags):
        if not bool(valid):
            continue
        target = rationale_targets[i]
        target = target * (target > 0).float()
        if target.sum() <= 0:
            continue
        target = target / target.sum()
        # Average cross-entropy across supervised heads.
        probs = attention[i].clamp_min(1e-8)
        per_head = -(target.unsqueeze(0) * probs.log()).sum(dim=-1)
        losses.append(per_head.mean())
    if not losses:
        return attention.sum() * 0.0
    return torch.stack(losses).mean()

def get_class_weights(frame, column, classes):
    counts = frame[column].value_counts()
    weights = compute_class_weight(
        class_weight="balanced", classes=np.array(classes),
        y=frame[column].astype(int).to_numpy()
    )
    return torch.tensor(weights, dtype=torch.float32, device=device)

tox_weights = get_class_weights(train_df, "label_id", [0, 1, 2])
sar_weights = get_class_weights(train_df, "sarcasm_id", [0, 1])
print("Toxicity class weights:", tox_weights)
print("Sarcasm class weights:", sar_weights)


Toxicity class weights: tensor([0.8904, 2.3953, 0.6852], device='cuda:0')
Sarcasm class weights: tensor([0.5814, 3.5728], device='cuda:0')


## 6. Training and validation-based checkpoint selection

The best checkpoint is selected by validation macro-F1. The test set is used only after training. For final claims, run all configured seeds and report mean ± standard deviation.

In [13]:
def predict_loader(model, loader, include_attention=False):
    model.eval()
    ys, ps, probs, sarcasm_ys, sarcasm_ps = [], [], [], [], []
    stored = []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tox_logits, sar_logits, attn = model(ids, mask)
            prob = torch.softmax(tox_logits, dim=-1)
            ys.extend(batch["label"].numpy().tolist())
            ps.extend(prob.argmax(-1).cpu().numpy().tolist())
            probs.extend(prob.cpu().numpy().tolist())
            sarcasm_ys.extend(batch["sarcasm"].numpy().tolist())
            sarcasm_ps.extend(torch.softmax(sar_logits, -1).argmax(-1).cpu().numpy().tolist())
            if include_attention:
                for i in range(len(batch["label"])):
                    stored.append({
                        "label": int(batch["label"][i]),
                        "sarcasm": int(batch["sarcasm"][i]),
                        "pred": int(prob[i].argmax().item()),
                        "probs": prob[i].cpu().numpy(),
                        "attention": attn[i].mean(0).cpu().numpy(),
                        "mask": batch["attention_mask"][i].numpy(),
                        "rationale_mask": batch["rationale_mask"][i].numpy(),
                        "rationale_valid": bool(batch["rationale_valid"][i]),
                        "text": batch["text"][i]
                    })
    out = {
        "y_true": np.array(ys),
        "y_pred": np.array(ps),
        "probs": np.array(probs),
        "sarcasm_true": np.array(sarcasm_ys),
        "sarcasm_pred": np.array(sarcasm_ps),
        "accuracy": accuracy_score(ys, ps),
        "macro_f1": f1_score(ys, ps, average="macro", zero_division=0)
    }
    if include_attention:
        out["details"] = stored
    return out

def train_one(seed, proposed, train_df, val_df, train_ds, val_ds,
              sarcasm_weight=None, rationale_weight=None):
    # Explicit parameters make each tuning run reproducible and auditable.
    sarcasm_weight = SARCASM_LOSS_WEIGHT if sarcasm_weight is None else float(sarcasm_weight)
    rationale_weight = RATIONALE_LOSS_WEIGHT if rationale_weight is None else float(rationale_weight)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_loader = make_loader(train_ds, shuffle=True)
    val_loader = make_loader(val_ds, shuffle=False)
    model = MultiTaskBERT().to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(0.1 * total_steps)),
        num_training_steps=total_steps
    )

    best_f1, best_state = -1.0, None
    history = []

    for epoch in range(EPOCHS):
        model.train()
        losses = {"total": 0.0, "tox": 0.0, "sar": 0.0, "rat": 0.0}
        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["label"].to(device)
            s = batch["sarcasm"].to(device)
            r = batch["rationale_mask"].to(device)
            valid = batch["rationale_valid"].tolist()

            optimizer.zero_grad(set_to_none=True)
            tox_logits, sar_logits, attn = model(ids, mask)
            loss_tox = F.cross_entropy(tox_logits, y, weight=tox_weights)
            loss_sar = F.cross_entropy(sar_logits, s, weight=sar_weights)
            loss_rat = rationale_loss_fn(attn, r, valid)

            if proposed:
                loss = loss_tox + sarcasm_weight * loss_sar + rationale_weight * loss_rat
            else:
                loss = loss_tox

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            losses["total"] += float(loss.detach().cpu())
            losses["tox"] += float(loss_tox.detach().cpu())
            losses["sar"] += float(loss_sar.detach().cpu())
            losses["rat"] += float(loss_rat.detach().cpu())

        val = predict_loader(model, val_loader)
        row = {
             "seed": seed, "model": "proposed" if proposed else "baseline",
            "sarcasm_weight": sarcasm_weight if proposed else 0.0,
            "rationale_weight": rationale_weight if proposed else 0.0,
            "epoch": epoch + 1,
            **{f"train_{k}": v / max(1, len(train_loader)) for k, v in losses.items()},
            "val_accuracy": val["accuracy"], "val_macro_f1": val["macro_f1"]
        }
        history.append(row)
        print(f"seed={seed} | {'PROPOSED' if proposed else 'BASELINE'} | "
              f"epoch={epoch+1}/{EPOCHS} | loss={row['train_total']:.4f} | "
              f"val macro-F1={val['macro_f1']:.4f}")

        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)



## 7. Train baseline and tune auxiliary-loss weights using validation only

For each seed, train the baseline and each proposed configuration. Each run keeps the checkpoint with the **highest validation macro-F1**. Then select the loss-weight configuration with the highest mean best-validation macro-F1 across seeds. The test set is not consulted during this selection.

This is a compact grid search, not a guarantee that the best possible weights are included. It can take time on CPU; GPU is recommended.


In [14]:
# Store CPU state dictionaries during the sweep to avoid keeping many BERT models on GPU.
baseline_states = {}
proposed_states_by_config = {cfg["name"]: {} for cfg in TUNING_CONFIGS}
histories = []
config_summary_rows = []

for seed in SEEDS:
    print("\n" + "=" * 70 + f"\nSEED {seed} — BASELINE\n" + "=" * 70)
    base_model, base_hist = train_one(seed, False, train_df, val_df, train_ds, val_ds)
    baseline_states[seed] = {k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()}
    histories.append(base_hist)
    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for cfg in TUNING_CONFIGS:
        print("\n" + "-" * 60)
        print(f"SEED {seed} — CONFIG {cfg['name']} | "
              f"sarcasm_weight={cfg['sarcasm_weight']} | "
              f"rationale_weight={cfg['rationale_weight']}")
        model, hist = train_one(
            seed, True, train_df, val_df, train_ds, val_ds,
            sarcasm_weight=cfg["sarcasm_weight"],
            rationale_weight=cfg["rationale_weight"]
        )
        # train_one returns the best validation checkpoint.
        proposed_states_by_config[cfg["name"]][seed] = {
            k: v.detach().cpu().clone() for k, v in model.state_dict().items()
        }
        best_row = hist.loc[hist["val_macro_f1"].idxmax()]
        config_summary_rows.append({
            "seed": seed,
            "config": cfg["name"],
            "sarcasm_weight": cfg["sarcasm_weight"],
            "rationale_weight": cfg["rationale_weight"],
            "best_epoch": int(best_row["epoch"]),
            "best_val_macro_f1": float(best_row["val_macro_f1"]),
            "best_val_accuracy": float(best_row["val_accuracy"])
        })
        histories.append(hist)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

history_df = pd.concat(histories, ignore_index=True)
history_df.to_csv("/kaggle/working/tuning_training_history.csv", index=False)
display(history_df)

config_results_df = pd.DataFrame(config_summary_rows)
display(config_results_df)

# Select using validation results only.
config_summary = (
    config_results_df.groupby(["config", "sarcasm_weight", "rationale_weight"], as_index=False)
    .agg(mean_best_val_macro_f1=("best_val_macro_f1", "mean"),
         std_best_val_macro_f1=("best_val_macro_f1", "std"),
         mean_best_val_accuracy=("best_val_accuracy", "mean"))
    .sort_values(["mean_best_val_macro_f1", "mean_best_val_accuracy"], ascending=False)
    .reset_index(drop=True)
)
print("Configurations ranked using validation macro-F1 (test set not used):")
display(config_summary)
config_summary.to_csv("/kaggle/working/validation_tuning_summary.csv", index=False)

best_config_name = config_summary.iloc[0]["config"]
best_config = next(cfg for cfg in TUNING_CONFIGS if cfg["name"] == best_config_name)
print("Selected configuration:", best_config)

# Rebuild only the chosen baseline and proposed models for test evaluation.
all_models = {}
for seed in SEEDS:
    base_model = MultiTaskBERT().to(device)
    base_model.load_state_dict(baseline_states[seed])
    all_models[(seed, "baseline")] = base_model

    proposed_model = MultiTaskBERT().to(device)
    proposed_model.load_state_dict(proposed_states_by_config[best_config_name][seed])
    all_models[(seed, "proposed")] = proposed_model

print("Selected configuration is stored as all_models[(seed, 'proposed')].")



SEED 42 — BASELINE


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

seed=42 | BASELINE | epoch=1/3 | loss=0.9701 | val macro-F1=0.5453
seed=42 | BASELINE | epoch=2/3 | loss=0.7259 | val macro-F1=0.6652
seed=42 | BASELINE | epoch=3/3 | loss=0.5081 | val macro-F1=0.6555

------------------------------------------------------------
SEED 42 — CONFIG sar_005_rat_0005 | sarcasm_weight=0.05 | rationale_weight=0.005
seed=42 | PROPOSED | epoch=1/3 | loss=1.0236 | val macro-F1=0.5396
seed=42 | PROPOSED | epoch=2/3 | loss=0.7783 | val macro-F1=0.6645
seed=42 | PROPOSED | epoch=3/3 | loss=0.5561 | val macro-F1=0.6521

------------------------------------------------------------
SEED 42 — CONFIG sar_005_rat_001 | sarcasm_weight=0.05 | rationale_weight=0.01
seed=42 | PROPOSED | epoch=1/3 | loss=1.0428 | val macro-F1=0.5600
seed=42 | PROPOSED | epoch=2/3 | loss=0.7959 | val macro-F1=0.6610
seed=42 | PROPOSED | epoch=3/3 | loss=0.5713 | val macro-F1=0.6492

------------------------------------------------------------
SEED 42 — CONFIG sar_01_rat_0005 | sarcasm_weight=0

,seed,model,sarcasm_weight,rationale_weight,epoch,train_total,train_tox,train_sar,train_rat,val_accuracy,val_macro_f1
0,42,baseline,0.00,0.000,1,0.970110,0.970110,0.699751,3.954102,0.670846,0.545347
1,42,baseline,0.00,0.000,2,0.725850,0.725850,0.722043,4.352340,0.739812,0.665190
2,42,baseline,0.00,0.000,3,0.508062,0.508062,0.721724,4.384900,0.736677,0.655537
3,42,proposed,0.05,0.005,1,1.023619,0.970457,0.681366,3.818825,0.673981,0.539589
4,42,proposed,0.05,0.005,2,0.778268,0.725121,0.667013,3.959266,0.742947,0.664477
5,42,proposed,0.05,0.005,3,0.556123,0.503506,0.657813,3.945242,0.733542,0.652071
6,42,proposed,0.05,0.010,1,1.042829,0.971425,0.681012,3.735344,0.689655,0.559973
7,42,proposed,0.05,0.010,2,0.795926,0.724984,0.666682,3.760786,0.739812,0.661027
8,42,proposed,0.05,0.010,3,0.571254,0.500999,0.659131,3.729801,0.730408,0.649185
9,42,proposed,0.10,0.005,1,1.056705,0.969642,0.679359,3.825353,0.686520,0.549287


,seed,config,sarcasm_weight,rationale_weight,best_epoch,best_val_macro_f1,best_val_accuracy
0,42,sar_005_rat_0005,0.05,0.005,2,0.664477,0.742947
1,42,sar_005_rat_001,0.05,0.010,2,0.661027,0.739812
2,42,sar_01_rat_0005,0.10,0.005,2,0.672787,0.742947
3,42,sar_01_rat_001,0.10,0.010,2,0.707602,0.768025
4,42,sar_01_rat_002,0.10,0.020,2,0.691775,0.752351
5,43,sar_005_rat_0005,0.05,0.005,2,0.656136,0.717868
6,43,sar_005_rat_001,0.05,0.010,3,0.666903,0.746082
7,43,sar_01_rat_0005,0.10,0.005,3,0.677780,0.755486
8,43,sar_01_rat_001,0.10,0.010,3,0.667649,0.749216
9,43,sar_01_rat_002,0.10,0.020,3,0.673938,0.749216


Configurations ranked using validation macro-F1 (test set not used):


,config,sarcasm_weight,rationale_weight,mean_best_val_macro_f1,std_best_val_macro_f1,mean_best_val_accuracy
0,sar_01_rat_001,0.10,0.010,0.681701,0.022458,0.759666
1,sar_01_rat_002,0.10,0.020,0.673163,0.019012,0.749216
2,sar_01_rat_0005,0.10,0.005,0.671948,0.006293,0.751306
3,sar_005_rat_0005,0.05,0.005,0.663476,0.006895,0.739812
4,sar_005_rat_001,0.05,0.010,0.658429,0.010028,0.741902


Selected configuration: {'name': 'sar_01_rat_001', 'sarcasm_weight': 0.1, 'rationale_weight': 0.01}
Selected configuration is stored as all_models[(seed, 'proposed')].


## 8. Test-set classification evaluation
Evaluate the baseline and the single configuration selected by validation macro-F1. Do not change the selected weights based on these test results.


In [15]:
def classification_metrics(out):
    metrics = {
        "accuracy": out["accuracy"],
        "macro_f1": out["macro_f1"]
    }
    try:
        y_oh = np.eye(3)[out["y_true"]]
        metrics["macro_auroc_ovr"] = roc_auc_score(
            y_oh, out["probs"], average="macro", multi_class="ovr"
        )
    except ValueError:
        metrics["macro_auroc_ovr"] = np.nan
    return metrics

test_loader = make_loader(test_ds, shuffle=False)
test_rows = []
test_outputs = {}

# Use each seed's best validation checkpoint, then evaluate exactly once on test.
for seed in SEEDS:
    for model_name in ["baseline", "proposed"]:
        model = all_models[(seed, model_name)]
        out = predict_loader(model, test_loader, include_attention=True)
        test_outputs[(seed, model_name)] = out
        row = {"seed": seed, "model": model_name, **classification_metrics(out)}
        test_rows.append(row)

        print(f"\\nSeed {seed} — {model_name}")
        print(classification_report(
            out["y_true"], out["y_pred"],
            labels=[0, 1, 2],
            target_names=["hatespeech", "offensive", "normal"],
            digits=4, zero_division=0
        ))
        print("Confusion matrix (true rows, predicted columns):")
        print(confusion_matrix(out["y_true"], out["y_pred"], labels=[0, 1, 2]))

test_metrics_df = pd.DataFrame(test_rows)
display(test_metrics_df)

summary = test_metrics_df.groupby("model")[["accuracy", "macro_f1", "macro_auroc_ovr"]].agg(["mean", "std"])
print("Mean and standard deviation across seeds:")
display(summary)

test_metrics_df.to_csv("/kaggle/working/test_classification_metrics_by_seed.csv", index=False)
summary.to_csv("/kaggle/working/test_classification_summary.csv")


\nSeed 42 — baseline
              precision    recall  f1-score   support

  hatespeech     0.8208    0.7311    0.7733       119
   offensive     0.2927    0.2727    0.2824        44
      normal     0.7558    0.8333    0.7927       156

    accuracy                         0.7179       319
   macro avg     0.6231    0.6124    0.6161       319
weighted avg     0.7162    0.7179    0.7151       319

Confusion matrix (true rows, predicted columns):
[[ 87  14  18]
 [  8  12  24]
 [ 11  15 130]]
\nSeed 42 — proposed
              precision    recall  f1-score   support

  hatespeech     0.8113    0.7227    0.7644       119
   offensive     0.2791    0.2727    0.2759        44
      normal     0.7588    0.8269    0.7914       156

    accuracy                         0.7116       319
   macro avg     0.6164    0.6074    0.6106       319
weighted avg     0.7122    0.7116    0.7102       319

Confusion matrix (true rows, predicted columns):
[[ 86  15  18]
 [  9  12  23]
 [ 11  16 129]]
\nSeed

,seed,model,accuracy,macro_f1,macro_auroc_ovr
0,42,baseline,0.717868,0.616123,0.824375
1,42,proposed,0.711599,0.610573,0.822480
2,43,baseline,0.736677,0.654114,0.817425
3,43,proposed,0.758621,0.657403,0.829289
4,44,baseline,0.733542,0.618849,0.826472
5,44,proposed,0.714734,0.609752,0.820955


Mean and standard deviation across seeds:


accuracy            macro_f1           macro_auroc_ovr          
              mean       std      mean       std            mean       std
model                                                                     
baseline  0.729363  0.010077  0.629695  0.021191        0.822757  0.004735
proposed  0.728318  0.026290  0.625909  0.027277        0.824242  0.004437

## 9. Explainability evaluation

### Plausibility
For test posts with matched manual sarcasm rationales, compare token-level attention against the annotated rationale using:
- token F1, where the number of selected top-attention tokens equals the number of rationale tokens;
- average precision / AUPRC over valid content tokens.

### Faithfulness
For the model's predicted class, measure:
- **Comprehensiveness:** probability(original) − probability(text with rationale tokens removed); higher is generally better.
- **Sufficiency:** probability(original) − probability(rationale-only text); lower is generally better.

These metrics are computed only on examples with matched sarcasm rationales. The deletion/retention evaluation is an approximation based on masking token embeddings; it should be described as such in a paper.

In [16]:
def rationale_metrics_for_details(details):
    token_f1s, aps = [], []
    comprehensiveness, sufficiency = [], []

    for d in details:
        if not d["rationale_valid"] or d["sarcasm"] != 1:
            continue

        target = d["rationale_mask"].astype(bool)
        valid_tokens = d["mask"].astype(bool)
        # Ignore special tokens at both ends.
        valid_positions = np.where(valid_tokens)[0]
        if len(valid_positions) >= 2:
            valid_tokens[valid_positions[0]] = False
            valid_tokens[valid_positions[-1]] = False
        target = target & valid_tokens

        scores = np.asarray(d["attention"], dtype=float)
        if target.sum() == 0 or valid_tokens.sum() == 0:
            continue

        y = target[valid_tokens].astype(int)
        s = scores[valid_tokens]

        k = int(target.sum())
        selected_local = np.argsort(s)[-k:]
        pred = np.zeros(len(y), dtype=bool)
        pred[selected_local] = True
        tp = np.logical_and(pred, y == 1).sum()
        precision = tp / max(pred.sum(), 1)
        recall = tp / max((y == 1).sum(), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        token_f1s.append(f1)

        if len(np.unique(y)) == 2:
            aps.append(average_precision_score(y, s))

        # Faithfulness via embedding-mask perturbation, calculated separately below.
    return {
        "rationale_examples": len(token_f1s),
        "token_f1": float(np.mean(token_f1s)) if token_f1s else np.nan,
        "token_auprc": float(np.mean(aps)) if aps else np.nan
    }

# Plausibility summary across seeds and models.
plausibility_rows = []
for (seed, model_name), out in test_outputs.items():
    met = rationale_metrics_for_details(out["details"])
    plausibility_rows.append({"seed": seed, "model": model_name, **met})

plausibility_df = pd.DataFrame(plausibility_rows)
display(plausibility_df)
plausibility_df.to_csv("/kaggle/working/test_rationale_plausibility.csv", index=False)


,seed,model,rationale_examples,token_f1,token_auprc
0,42,baseline,43,0.263327,0.332876
1,42,proposed,43,0.348503,0.418856
2,43,baseline,43,0.249572,0.339873
3,43,proposed,43,0.303212,0.390281
4,44,baseline,43,0.244093,0.337048
5,44,proposed,43,0.350130,0.433430


In [17]:
def evaluate_faithfulness(model, dataset, max_examples=300):
    model.eval()
    rows = []
    embedding_layer = model.encoder.get_input_embeddings()

    # Restrict to test examples with matched sarcasm rationales.
    candidates = [
        x for x in dataset.rows
        if x["sarcasm"] == 1 and x["rationale_valid"]
    ][:max_examples]

    for item in candidates:
        ids = item["input_ids"].unsqueeze(0).to(device)
        mask = item["attention_mask"].unsqueeze(0).to(device)
        rationale = item["rationale_mask"].unsqueeze(0).to(device).bool()

        # Keep only non-special, non-padding tokens in the rationale.
        special_ids = set(tokenizer.all_special_ids)
        token_ids = ids[0].tolist()
        non_special = torch.tensor(
            [[(tid not in special_ids) for tid in token_ids]],
            device=device, dtype=torch.bool
        )
        rationale = rationale & non_special & mask.bool()
        if rationale.sum() == 0:
            continue

        with torch.no_grad():
            original_emb = embedding_layer(ids)
            original_logits = model(
                ids, mask
            )[0]
            pred_class = original_logits.argmax(-1)
            p_original = torch.softmax(original_logits, -1)[0, pred_class].item()

            # Comprehensiveness: remove rationale tokens by replacing their embeddings
            # with the [PAD] embedding while retaining the attention mask.
            pad_id = tokenizer.pad_token_id
            pad_emb = embedding_layer(
                torch.full_like(ids, pad_id)
            )
            removed_emb = torch.where(rationale.unsqueeze(-1), pad_emb, original_emb)
            removed_logits = model.encoder(
                inputs_embeds=removed_emb,
                attention_mask=mask,
                output_attentions=False,
                return_dict=True
            ).last_hidden_state[:, 0]
            removed_logits = model.dropout(removed_logits)
            removed_logits = model.toxicity_head(removed_logits)
            p_removed = torch.softmax(removed_logits, -1)[0, pred_class].item()

            # Sufficiency: retain rationale tokens; replace other content-token embeddings.
            only_rat_emb = torch.where(
                (non_special & mask.bool() & ~rationale).unsqueeze(-1),
                pad_emb, original_emb
            )
            only_logits = model.encoder(
                inputs_embeds=only_rat_emb,
                attention_mask=mask,
                output_attentions=False,
                return_dict=True
            ).last_hidden_state[:, 0]
            only_logits = model.dropout(only_logits)
            only_logits = model.toxicity_head(only_logits)
            p_only = torch.softmax(only_logits, -1)[0, pred_class].item()

        rows.append({
            "comprehensiveness": p_original - p_removed,
            "sufficiency": p_original - p_only
        })

    return pd.DataFrame(rows)

faithfulness_rows = []
for seed in SEEDS:
    for model_name in ["baseline", "proposed"]:
        print("Faithfulness:", seed, model_name)
        faith_df = evaluate_faithfulness(
            all_models[(seed, model_name)], test_ds
        )
        faithfulness_rows.append({
            "seed": seed, "model": model_name,
            "n": len(faith_df),
            "comprehensiveness_mean": faith_df["comprehensiveness"].mean(),
            "sufficiency_mean": faith_df["sufficiency"].mean()
        })

faithfulness_df = pd.DataFrame(faithfulness_rows)
display(faithfulness_df)
faithfulness_df.to_csv("/kaggle/working/test_rationale_faithfulness.csv", index=False)


Faithfulness: 42 baseline
Faithfulness: 42 proposed
Faithfulness: 43 baseline
Faithfulness: 43 proposed
Faithfulness: 44 baseline
Faithfulness: 44 proposed


,seed,model,n,comprehensiveness_mean,sufficiency_mean
0,42,baseline,43,0.016297,0.247171
1,42,proposed,43,0.004240,0.226625
2,43,baseline,43,0.034477,0.333870
3,43,proposed,43,0.058185,0.323967
4,44,baseline,43,0.062346,0.242595
5,44,proposed,43,0.061965,0.242260


## 10. Statistical comparison and result export

Use paired bootstrap resampling on the held-out test predictions for a confidence interval on the proposed-minus-baseline macro-F1 difference. With multiple seeds, treat this as supporting evidence rather than a replacement for a pre-registered statistical design.

In [18]:
def macro_f1_from_arrays(y, pred):
    return f1_score(y, pred, average="macro", zero_division=0)

def paired_bootstrap_f1(y, pred_base, pred_prop, n_boot=2000, seed=123):
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        # Skip degenerate resamples missing one or more classes.
        if len(np.unique(y[idx])) < 3:
            continue
        f1_b = macro_f1_from_arrays(y[idx], pred_base[idx])
        f1_p = macro_f1_from_arrays(y[idx], pred_prop[idx])
        diffs.append(f1_p - f1_b)
    if not diffs:
        return np.nan, (np.nan, np.nan)
    return float(np.mean(diffs)), tuple(np.percentile(diffs, [2.5, 97.5]))

# Bootstrap each seed on the exact same held-out examples.
bootstrap_rows = []
for seed in SEEDS:
    b = test_outputs[(seed, "baseline")]
    p = test_outputs[(seed, "proposed")]
    mean_diff, ci = paired_bootstrap_f1(b["y_true"], b["y_pred"], p["y_pred"])
    bootstrap_rows.append({
        "seed": seed,
        "macro_f1_difference_proposed_minus_baseline": mean_diff,
        "ci95_low": ci[0],
        "ci95_high": ci[1]
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)
bootstrap_df.to_csv("/kaggle/working/paired_bootstrap_macro_f1.csv", index=False)

# Save a consolidated results workbook.
output_path = "/kaggle/working/sarcasm_rationale_experiment_results.xlsx"
with pd.ExcelWriter(output_path) as writer:
    test_metrics_df.to_excel(writer, sheet_name="classification_by_seed", index=False)
    summary.to_excel(writer, sheet_name="classification_summary")
    plausibility_df.to_excel(writer, sheet_name="rationale_plausibility", index=False)
    faithfulness_df.to_excel(writer, sheet_name="rationale_faithfulness", index=False)
    bootstrap_df.to_excel(writer, sheet_name="bootstrap_f1", index=False)
    history_df.to_excel(writer, sheet_name="training_history", index=False)
    config_results_df.to_excel(writer, sheet_name="tuning_by_seed", index=False)
    config_summary.to_excel(writer, sheet_name="tuning_summary", index=False)

print("Saved results to:", output_path)


,seed,macro_f1_difference_proposed_minus_baseline,ci95_low,ci95_high
0,42,-0.005536,-0.018060,0.005583
1,43,0.003450,-0.041420,0.046184
2,44,-0.008478,-0.028527,0.015638


Saved results to: /kaggle/working/sarcasm_rationale_experiment_results.xlsx


## 11. How to interpret and report the results

1. Report dataset distribution and the number of examples with successfully aligned rationale spans.
2. Report the validation-selected auxiliary-loss weights and mean validation macro-F1 across seeds.
3. Report held-out test accuracy, macro-F1, AUROC, per-class metrics, rationale token F1/AUPRC, and faithfulness approximation metrics.
4. Compare the selected proposed configuration against the baseline. The tuning sweep is selected on validation only; do not select a configuration after looking at test metrics.
5. If rationale quality improves but macro-F1 still falls, report the trade-off honestly and expand the validation-only search in a later experiment.

Outputs are saved under `/kaggle/working/`, including `sarcasm_rationale_experiment_results.xlsx`, `validation_tuning_summary.csv`, and `tuning_training_history.csv`.
